In [ ]:
import json
import pandas as pd
from google.colab import files

uploaded = files.upload()

assert len(uploaded) == 1, "Please upload only the source JSON file."

source_filename = next(iter(uploaded))
source_data = json.loads(
    uploaded[source_filename].decode("utf-8")
)

print("Dataset loaded.")
print("Machines:", len(source_data["machines"]))
print("Products:", len(source_data["products"]))

In [ ]:
SLOT_MINUTES = 5
WORKDAY_START_MINUTES = 7 * 60
WORKDAY_END_MINUTES = 23 * 60

SLOTS_PER_DAY = (
    WORKDAY_END_MINUTES - WORKDAY_START_MINUTES
) // SLOT_MINUTES

grid_source = next(
    source
    for source in source_data["energy_sources"]
    if source["id"] == "Socket Energy"
)

all_prices_eur_per_wh = grid_source["price"]

# Check that the data contains complete working days.
assert len(all_prices_eur_per_wh) % SLOTS_PER_DAY == 0

daily_prices = [
    all_prices_eur_per_wh[start:start + SLOTS_PER_DAY]
    for start in range(
        0, len(all_prices_eur_per_wh), SLOTS_PER_DAY
    )
]

price_summary_rows = []

for day_number, prices in enumerate(daily_prices, start=1):
    prices_eur_per_kwh = [
        price * 1000
        for price in prices
    ]

    price_summary_rows.append({
        "day": day_number,
        "time_slots": len(prices),
        "min_price_eur_kwh": min(prices_eur_per_kwh),
        "mean_price_eur_kwh": (
            sum(prices_eur_per_kwh) / len(prices_eur_per_kwh)
        ),
        "max_price_eur_kwh": max(prices_eur_per_kwh),
    })

daily_price_summary_df = pd.DataFrame(price_summary_rows)

display(daily_price_summary_df)

print("Working days:", len(daily_prices))
print("Time slots per day:", SLOTS_PER_DAY)

In [ ]:
# Select the three products and their processing modes.
product_selection = [
    ("J1", "PRINTED RIBBON  PES", "Sublimation TM3"),
    ("J2", "BRAIDED ELASTIC", "Anti-Shrinkage TM2"),
    ("J3", "SMOOTH STRAP(A)", "Ironing TM1"),
]

products_by_id = {
    product["id"]: product
    for product in source_data["products"]
}

tasks_by_id = {
    task["id"]: task
    for task in source_data["tasks"]
}

modes_by_id = {
    mode["id"]: mode
    for mode in source_data["task_modes"]
}

study_machine = next(
    machine
    for machine in source_data["machines"]
    if machine["id"] == "MAQ120"
)

job_types = []

for job_id, product_name, mode_name in product_selection:
    product = products_by_id[product_name]

    # Only include products requiring one task, performed once.
    assert len(product["tasks"]) == 1
    task_entry = product["tasks"][0]
    assert task_entry["runs"] == 1

    # Verify the task, mode and machine relationship.
    task = tasks_by_id[task_entry["task"]]
    assert mode_name in task["task_modes"]
    assert mode_name in study_machine["task_modes"]

    quantity = sum(
        request["amount"]
        for request in source_data["product_requests"]
        if request["product"] == product_name
    )

    assert quantity > 0 and quantity == int(quantity)

    profile_wh = list(modes_by_id[mode_name]["power"])

    job_types.append({
        "job_type_id": job_id,
        "product": product_name,
        "mode": mode_name,
        "quantity": int(quantity),
        "profile_wh": profile_wh,
        "duration_slots": len(profile_wh),
        "duration_minutes": len(profile_wh) * SLOT_MINUTES,
        "energy_kwh_per_unit": sum(profile_wh) / 1000,
    })

job_types_df = pd.DataFrame(job_types)

display(
    job_types_df.drop(columns=["profile_wh"])
)

total_processing_minutes = sum(
    job["quantity"] * job["duration_minutes"]
    for job in job_types
)

total_processing_energy_kwh = sum(
    job["quantity"] * job["energy_kwh_per_unit"]
    for job in job_types
)

available_minutes = (
    WORKDAY_END_MINUTES - WORKDAY_START_MINUTES
)

print("Total production units:", sum(
    job["quantity"] for job in job_types
))
print("Production time:", total_processing_minutes, "min")
print(f"Production energy: {total_processing_energy_kwh:.3f} kWh")
print(
    "Time available for setup and waiting:",
    available_minutes - total_processing_minutes,
    "min",
)

assert total_processing_minutes == 905
assert abs(total_processing_energy_kwh - 56.655) < 0.00000001

print("Input checks passed.")

In [ ]:
from functools import lru_cache
from math import inf, isfinite


def optimize_day(
    job_types,
    prices_eur_per_wh,
    setup_minutes=5,
    slot_minutes=5,
):
    # Validate and copy the inputs.
    assert isinstance(slot_minutes, int) and slot_minutes > 0
    assert setup_minutes >= 0
    assert setup_minutes % slot_minutes == 0
    assert job_types and prices_eur_per_wh

    prices = tuple(float(p) for p in prices_eur_per_wh)
    profiles = tuple(
        tuple(job["profile_wh"]) for job in job_types
    )
    quantities = tuple(job["quantity"] for job in job_types)

    assert all(isfinite(p) for p in prices)
    assert all(
        isinstance(q, int) and q >= 0 for q in quantities
    )
    assert all(
        profile and all(isfinite(e) and e >= 0 for e in profile)
        for profile in profiles
    )

    durations = tuple(len(profile) for profile in profiles)
    horizon = len(prices)
    preparation_slots = int(setup_minutes / slot_minutes)

    # Cost of starting each product type at each possible time.
    start_costs = []

    for profile in profiles:
        start_costs.append(tuple(
            sum(
                energy * prices[start + offset]
                for offset, energy in enumerate(profile)
            )
            if start + len(profile) <= horizon else inf
            for start in range(horizon + 1)
        ))

    # Find the minimum cost under the production constraints.
    @lru_cache(maxsize=None)
    def solve(time_slot, remaining, last_type):
        if not any(remaining):
            return 0.0

        required_slots = sum(
            q * d for q, d in zip(remaining, durations)
        )

        if time_slot + required_slots > horizon:
            return inf

        best_cost = solve(
            time_slot + 1, remaining, last_type
        )

        for kind, quantity in enumerate(remaining):
            if quantity == 0:
                continue

            change_required = (
                last_type != -1 and last_type != kind
            )

            start = time_slot + (
                preparation_slots if change_required else 0
            )
            finish = start + durations[kind]

            if finish > horizon:
                continue

            next_remaining = list(remaining)
            next_remaining[kind] -= 1

            candidate_cost = (
                start_costs[kind][start]
                + solve(
                    finish, tuple(next_remaining), kind
                )
            )

            best_cost = min(best_cost, candidate_cost)

        return best_cost

    optimal_cost = solve(0, quantities, -1)

    # Reference: original product order, no deliberate waiting.
    time_slot = 0
    previous_type = -1
    reference_cost = 0.0

    for kind, quantity in enumerate(quantities):
        for _ in range(quantity):
            if previous_type != -1 and previous_type != kind:
                time_slot += preparation_slots

            if time_slot + durations[kind] > horizon:
                reference_cost = inf
                break

            reference_cost += start_costs[kind][time_slot]
            time_slot += durations[kind]
            previous_type = kind

        if not isfinite(reference_cost):
            break

    comparable = (
        isfinite(optimal_cost) and isfinite(reference_cost)
    )

    if comparable:
        assert optimal_cost <= reference_cost + 0.000000001

    result = {
        "setup_minutes": setup_minutes,
        "optimized_feasible": isfinite(optimal_cost),
        "reference_feasible": isfinite(reference_cost),
        "reference_cost_eur": (
            reference_cost if isfinite(reference_cost) else None
        ),
        "optimized_cost_eur": (
            optimal_cost if isfinite(optimal_cost) else None
        ),
        "savings_eur": (
            reference_cost - optimal_cost
            if comparable else None
        ),
        "savings_percent": (
            (reference_cost - optimal_cost)
            / reference_cost * 100
            if comparable and reference_cost > 0 else None
        ),
        "states_evaluated": solve.cache_info().currsize,
    }

    solve.cache_clear()
    return result

In [ ]:
first_day_check = optimize_day(
    job_types=job_types,
    prices_eur_per_wh=daily_prices[0],
    setup_minutes=5,
)

display(pd.DataFrame([first_day_check]))

assert first_day_check["optimized_feasible"]

assert abs(
    first_day_check["reference_cost_eur"] - 3.86126973
) < 0.00000001

assert abs(
    first_day_check["optimized_cost_eur"] - 3.79714186
) < 0.00000001

print("Check passed: the new function matches the first notebook.")

In [ ]:
daily_results = []

for day_number, prices in enumerate(daily_prices, start=1):
    result = optimize_day(
        job_types=job_types,
        prices_eur_per_wh=prices,
        setup_minutes=5,
    )

    daily_results.append({
        "day": day_number,
        **result,
    })

daily_comparison_df = pd.DataFrame(daily_results)

# Check that all six scenarios can be compared.
assert len(daily_comparison_df) == 6
assert daily_comparison_df["optimized_feasible"].all()
assert daily_comparison_df["reference_feasible"].all()

display(
    daily_comparison_df[[
        "day",
        "reference_cost_eur",
        "optimized_cost_eur",
        "savings_eur",
        "savings_percent",
    ]].style.format({
        "reference_cost_eur": "{:.6f}",
        "optimized_cost_eur": "{:.6f}",
        "savings_eur": "{:.6f}",
        "savings_percent": "{:.2f}",
    })
)

print("All six daily scenarios completed.")

In [ ]:
total_reference = daily_comparison_df[
    "reference_cost_eur"
].sum()

total_optimized = daily_comparison_df[
    "optimized_cost_eur"
].sum()

total_savings = total_reference - total_optimized
overall_savings_percent = (
    total_savings / total_reference * 100
)

print(f"Total reference cost: {total_reference:.6f} EUR")
print(f"Total optimized cost: {total_optimized:.6f} EUR")
print(f"Total savings: {total_savings:.6f} EUR")
print(f"Overall savings: {overall_savings_percent:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    daily_comparison_df["day"],
    daily_comparison_df["savings_percent"],
    color="#2563EB",
)

for bar in bars:
    height = bar.get_height()

    ax.annotate(
        f"{height:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
    )

ax.set_xticks(daily_comparison_df["day"])
ax.set_xlabel("Price day")
ax.set_ylabel("Electricity cost savings (%)")
ax.set_title(
    "Daily savings with identical demand and 5-minute setups"
)
ax.set_ylim(
    0,
    daily_comparison_df["savings_percent"].max() * 1.25,
)

ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig("daily_savings_comparison.png", dpi=300)
plt.show()

In [ ]:
first_day_mean_price = (
    sum(daily_prices[0]) / len(daily_prices[0])
)

flat_prices = [
    first_day_mean_price
] * len(daily_prices[0])

flat_price_result = optimize_day(
    job_types=job_types,
    prices_eur_per_wh=flat_prices,
    setup_minutes=5,
)

print(
    f"Reference cost: "
    f"{flat_price_result['reference_cost_eur']:.8f} EUR"
)

print(
    f"Optimized cost: "
    f"{flat_price_result['optimized_cost_eur']:.8f} EUR"
)

print(
    f"Savings: "
    f"{flat_price_result['savings_percent']:.6f}%"
)

assert abs(flat_price_result["savings_eur"]) < 0.00000001

print("Check passed: constant prices produce no cost savings.")

In [ ]:
# Keep product proportions constant across workload levels.
mix_by_job_id = {
    "J1": 2,
    "J2": 3,
    "J3": 1,
}

capacity_rows = []
available_minutes = SLOTS_PER_DAY * SLOT_MINUTES

for level in range(1, 7):
    # Create new inputs without changing the original job_types.
    scenario_jobs = [
        {
            **job,
            "quantity": mix_by_job_id[job["job_type_id"]] * level,
        }
        for job in job_types
    ]

    processing_minutes = sum(
        job["quantity"]
        * len(job["profile_wh"])
        * SLOT_MINUTES
        for job in scenario_jobs
    )

    result = optimize_day(
        job_types=scenario_jobs,
        prices_eur_per_wh=daily_prices[0],
        setup_minutes=5,
    )

    capacity_rows.append({
        "workload_level": level,
        "processing_minutes": processing_minutes,
        "processing_utilization_percent": (
            processing_minutes / available_minutes * 100
        ),
        **result,
    })

capacity_comparison_df = pd.DataFrame(capacity_rows)

display(
    capacity_comparison_df[[
        "workload_level",
        "processing_minutes",
        "processing_utilization_percent",
        "reference_cost_eur",
        "optimized_cost_eur",
        "savings_percent",
        "optimized_feasible",
    ]].style.format({
        "processing_utilization_percent": "{:.2f}",
        "reference_cost_eur": "{:.6f}",
        "optimized_cost_eur": "{:.6f}",
        "savings_percent": "{:.2f}",
    }, na_rep="Not feasible")
)

assert capacity_comparison_df.iloc[-1][
    "optimized_feasible"
] == False

print("Capacity scenarios completed.")

In [ ]:
import matplotlib.pyplot as plt

# Only plot scenarios with valid comparison results.
feasible_capacity_df = capacity_comparison_df.loc[
    capacity_comparison_df["optimized_feasible"]
    & capacity_comparison_df["reference_feasible"]
].copy()

x = feasible_capacity_df["processing_utilization_percent"]

fig, axes = plt.subplots(
    1, 2, figsize=(12, 4.5), sharex=True
)

plot_settings = [
    ("savings_percent", "Electricity cost savings (%)", "{:.2f}%"),
    ("savings_eur", "Electricity cost savings (EUR)", "{:.4f}"),
]

for ax, (column, ylabel, label_format) in zip(
    axes, plot_settings
):
    y = feasible_capacity_df[column]

    ax.plot(x, y, marker="o", color="#2563EB")

    for utilization, value in zip(x, y):
        ax.annotate(
            label_format.format(value),
            xy=(utilization, value),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=9,
        )

    ax.set_xlabel("Processing utilization (%)")
    ax.set_ylabel(ylabel)
    ax.set_xlim(10, 105)
    ax.set_ylim(0, y.max() * 1.25)
    ax.grid(alpha=0.2)

fig.suptitle(
    "Workload sensitivity — price day 1, 5-minute setups"
)

fig.tight_layout()
fig.savefig("capacity_savings_comparison.png", dpi=300)
plt.show()

capacity_comparison_df.to_csv(
    "capacity_sensitivity_results.csv",
    index=False,
)

In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile("capacity_analysis.zip", "w") as archive:
    archive.write("capacity_savings_comparison.png")
    archive.write("capacity_sensitivity_results.csv")

files.download("capacity_analysis.zip")

In [ ]:
capacity_day_rows = []

# Repeat each workload level under all six price profiles.
for day_number, prices in enumerate(daily_prices, start=1):
    for level in range(1, 6):

        scenario_jobs = [
            {
                **job,
                "quantity": (
                    mix_by_job_id[job["job_type_id"]] * level
                ),
            }
            for job in job_types
        ]

        processing_minutes = sum(
            job["quantity"]
            * len(job["profile_wh"])
            * SLOT_MINUTES
            for job in scenario_jobs
        )

        result = optimize_day(
            job_types=scenario_jobs,
            prices_eur_per_wh=prices,
            setup_minutes=5,
        )

        capacity_day_rows.append({
            "day": day_number,
            "workload_level": level,
            "processing_minutes": processing_minutes,
            "processing_utilization_percent": (
                processing_minutes
                / (SLOTS_PER_DAY * SLOT_MINUTES)
                * 100
            ),
            **result,
        })

    print(f"Day {day_number}: five workload levels completed.")

capacity_day_results_df = pd.DataFrame(capacity_day_rows)

assert len(capacity_day_results_df) == 30
assert capacity_day_results_df["optimized_feasible"].all()
assert capacity_day_results_df["reference_feasible"].all()

print("\nAll 30 scenarios completed successfully.")

In [ ]:
capacity_summary_df = (
    capacity_day_results_df
    .groupby("workload_level", as_index=False)
    .agg(
        processing_utilization_percent=(
            "processing_utilization_percent", "first"
        ),
        total_reference_cost_eur=("reference_cost_eur", "sum"),
        total_optimized_cost_eur=("optimized_cost_eur", "sum"),
        min_daily_savings_percent=("savings_percent", "min"),
        max_daily_savings_percent=("savings_percent", "max"),
    )
)

capacity_summary_df["combined_savings_percent"] = (
    (
        capacity_summary_df["total_reference_cost_eur"]
        - capacity_summary_df["total_optimized_cost_eur"]
    )
    / capacity_summary_df["total_reference_cost_eur"]
    * 100
)

display(
    capacity_summary_df[[
        "processing_utilization_percent",
        "combined_savings_percent",
        "min_daily_savings_percent",
        "max_daily_savings_percent",
    ]].round(2)
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Arrange the 30 results into a 5-by-6 table.
heatmap_table = (
    capacity_day_results_df.pivot(
        index="processing_utilization_percent",
        columns="day",
        values="savings_percent",
    )
    .sort_index()
    .sort_index(axis=1)
)

assert heatmap_table.shape == (5, 6)
assert not heatmap_table.isna().any().any()

values = heatmap_table.to_numpy()

fig, ax = plt.subplots(figsize=(10, 5))

image = ax.imshow(
    values,
    cmap="YlGnBu",
    aspect="auto",
    vmin=0,
    vmax=values.max(),
)

ax.set_xticks(np.arange(len(heatmap_table.columns)))
ax.set_xticklabels([
    f"Day {day}" for day in heatmap_table.columns
])

ax.set_yticks(np.arange(len(heatmap_table.index)))
ax.set_yticklabels([
    f"{utilization:.2f}%"
    for utilization in heatmap_table.index
])

# Write each scenario's savings inside its cell.
for row in range(values.shape[0]):
    for column in range(values.shape[1]):
        value = values[row, column]
        colour = image.cmap(image.norm(value))

        brightness = (
            0.2126 * colour[0]
            + 0.7152 * colour[1]
            + 0.0722 * colour[2]
        )

        ax.text(
            column,
            row,
            f"{value:.2f}%",
            ha="center",
            va="center",
            color="white" if brightness < 0.5 else "black",
        )

ax.set_xlabel("Price day")
ax.set_ylabel("Processing utilization")
ax.set_title(
    "Electricity cost savings by workload and price day\n"
    "Fixed product mix 2:3:1 — 5-minute setups"
)

colourbar = fig.colorbar(image, ax=ax)
colourbar.set_label("Savings versus reference plan (%)")

fig.tight_layout()
fig.savefig("workload_price_day_heatmap.png", dpi=300)
plt.show()

In [ ]:
import zipfile
from google.colab import files

capacity_day_results_df.to_csv(
    "workload_price_day_results.csv", index=False
)

capacity_summary_df.to_csv(
    "workload_summary.csv", index=False
)

with zipfile.ZipFile("workload_price_analysis.zip", "w") as archive:
    archive.write("workload_price_day_heatmap.png")
    archive.write("workload_price_day_results.csv")
    archive.write("workload_summary.csv")

files.download("workload_price_analysis.zip")

## Findings and interpretation

The workload analysis compared 30 scenarios combining five workload levels with six electricity price profiles. All scenarios used machine MAQ120, a fixed product mix of 2:3:1 and an assumed five-minute setup whenever production switched between product types.

At 19.79% processing utilization, combined electricity cost savings across the six price profiles were 6.53%. At 98.96% processing utilization, combined savings fell to 0.29%. Lower workloads generally provided greater flexibility to move production into cheaper periods.

The relationship was not strictly decreasing. Combined savings were 1.94% at 59.38% utilization and 2.25% at 79.17% utilization. Scheduling benefits therefore depended on the interaction between workload, processing profiles and the timing of electricity prices.

Savings also varied between price days. At the lowest workload, daily savings ranged from 3.91% to 8.34%. At the highest feasible workload, they ranged from 0.09% to 0.56%.

## Scope and limitations

These are modelling scenarios based on published production and electricity-price data, not measured savings from an implemented factory intervention. The same workload was tested independently against each price day.

Processing utilization excludes setup time. Setup and waiting energy consumption were assumed to be zero. All jobs were available at 07:00, production had to finish by 23:00, and complete daily prices were assumed to be known before scheduling. Jobs could not be interrupted.

The reference plan produced each product type consecutively in a fixed order, starting at 07:00. Savings are relative to this reference and do not establish an advantage over every alternative scheduling policy.

The results describe electricity cost reductions, not reductions in production energy, total operating costs or verified annual expenditure.

In [ ]:
from itertools import permutations
from math import inf


def grouped_order_cost(
    ordered_jobs,
    prices,
    setup_minutes=5,
    slot_minutes=5,
):
    preparation_slots = setup_minutes // slot_minutes
    current_slot = 0
    previous_mode = None
    total_cost = 0.0

    for job in ordered_jobs:
        if job["quantity"] == 0:
            continue

        if (
            previous_mode is not None
            and previous_mode != job["mode"]
        ):
            current_slot += preparation_slots

        profile = job["profile_wh"]

        for _ in range(job["quantity"]):
            if current_slot + len(profile) > len(prices):
                return inf

            total_cost += sum(
                energy * prices[current_slot + offset]
                for offset, energy in enumerate(profile)
            )

            current_slot += len(profile)

        previous_mode = job["mode"]

    return total_cost


grouped_order_rows = []

for order in permutations(job_types):
    grouped_order_rows.append({
        "product_order": " → ".join(
            job["job_type_id"] for job in order
        ),
        "cost_eur": grouped_order_cost(
            ordered_jobs=order,
            prices=daily_prices[0],
            setup_minutes=5,
        ),
    })

grouped_order_comparison_df = (
    pd.DataFrame(grouped_order_rows)
    .sort_values("cost_eur")
    .reset_index(drop=True)
)

display(grouped_order_comparison_df)

best_grouped_cost = grouped_order_comparison_df.iloc[0]["cost_eur"]
flexible_cost = first_day_check["optimized_cost_eur"]

additional_savings_percent = (
    (best_grouped_cost - flexible_cost)
    / best_grouped_cost
    * 100
)

print(f"Best grouped plan: {best_grouped_cost:.8f} EUR")
print(f"Flexible optimized plan: {flexible_cost:.8f} EUR")
print(
    f"Additional savings versus best grouped plan: "
    f"{additional_savings_percent:.2f}%"
)

assert abs(best_grouped_cost - 3.84003163) < 0.00000001
assert flexible_cost <= best_grouped_cost + 0.00000001

In [ ]:
benchmark_rows = []

for day_number, prices in enumerate(daily_prices, start=1):
    grouped_candidates = []

    for order in permutations(job_types):
        grouped_candidates.append({
            "order": " → ".join(
                job["job_type_id"] for job in order
            ),
            "cost_eur": grouped_order_cost(
                ordered_jobs=order,
                prices=prices,
                setup_minutes=5,
            ),
        })

    best_grouped = min(
        grouped_candidates,
        key=lambda candidate: candidate["cost_eur"],
    )

    # Retrieve the flexible result already calculated for this day.
    matching_day = daily_comparison_df.loc[
        daily_comparison_df["day"] == day_number
    ]

    assert len(matching_day) == 1
    original_result = matching_day.iloc[0]

    flexible_cost = original_result["optimized_cost_eur"]
    grouped_cost = best_grouped["cost_eur"]

    assert flexible_cost <= grouped_cost + 0.00000001

    benchmark_rows.append({
        "day": day_number,
        "best_grouped_order": best_grouped["order"],
        "original_reference_cost_eur": (
            original_result["reference_cost_eur"]
        ),
        "best_grouped_cost_eur": grouped_cost,
        "flexible_cost_eur": flexible_cost,
        "additional_savings_eur": grouped_cost - flexible_cost,
        "additional_savings_percent": (
            (grouped_cost - flexible_cost)
            / grouped_cost * 100
        ),
    })

benchmark_comparison_df = pd.DataFrame(benchmark_rows)

display(
    benchmark_comparison_df.style.format({
        "original_reference_cost_eur": "{:.6f}",
        "best_grouped_cost_eur": "{:.6f}",
        "flexible_cost_eur": "{:.6f}",
        "additional_savings_eur": "{:.6f}",
        "additional_savings_percent": "{:.2f}",
    })
)

total_grouped_cost = benchmark_comparison_df[
    "best_grouped_cost_eur"
].sum()

total_flexible_cost = benchmark_comparison_df[
    "flexible_cost_eur"
].sum()

combined_additional_savings = (
    (total_grouped_cost - total_flexible_cost)
    / total_grouped_cost * 100
)

print(
    f"Combined savings versus best grouped plans: "
    f"{combined_additional_savings:.2f}%"
)

benchmark_comparison_df.to_csv(
    "grouped_vs_flexible_comparison.csv",
    index=False,
)

## Comparison against the best grouped production order

An additional benchmark tested all six grouped product orders for each price day. Each grouped plan started at 07:00, included five-minute product-change setups and produced all units of a product consecutively, without deliberate waiting.

This comparison used the original selected quantities: 10 PES ribbon units, 15 braided elastic units and 4 smooth strap units. These quantities differ from the fixed 2:3:1 product mix used in the workload sensitivity analysis.

Across the six independent daily scenarios, the best grouped plans cost 22.649091 EUR in total. The flexible optimized plans cost 22.371789 EUR, a reduction of 0.277302 EUR or 1.22%. Daily reductions relative to the best grouped plan ranged from 0.75% to 1.76%.

Against the original fixed-order reference, the combined reduction was 1.50%. The smaller advantage against the improved benchmark shows that benchmark selection affects the reported benefit. Nevertheless, the flexible model retained a cost advantage in all six tested price profiles.

The grouped benchmark also used electricity prices to select its order. Therefore, this comparison does not isolate the value of price information. It measures the combined additional value of allowing product groups to be split and production timing to be adjusted, relative to grouped plans starting at 07:00.

## Practical interpretation

The calculated monetary savings are small for the selected production system and price profiles. Percentage improvements alone do not establish that implementation would be economically worthwhile.

A practical implementation assessment would also require verified setup durations, setup and idle energy consumption, labour implications, electricity contract terms and implementation costs. Annual savings cannot be inferred reliably from these six price profiles alone.

In [ ]:
from itertools import permutations, combinations
from math import inf, isfinite

# Small artificial example: three jobs, one unit each.
test_jobs = [
    {"quantity": 1, "profile_wh": [10, 30]},
    {"quantity": 1, "profile_wh": [20]},
    {"quantity": 1, "profile_wh": [12, 8]},
]

test_prices = [
    0.00008, 0.00009, 0.00004,
    0.00003, 0.00007, 0.00005,
    0.00010, 0.00002, 0.00006,
]


def exhaustive_minimum(jobs, prices, setup_minutes):
    best_cost = inf
    valid_plans = 0
    setup_slots = setup_minutes // 5

    # Try every job order and every increasing set of start times.
    for order in permutations(range(len(jobs))):
        for starts in combinations(
            range(len(prices)), len(jobs)
        ):
            previous_finish = 0
            total_cost = 0.0

            for position, job_type in enumerate(order):
                start = starts[position]
                profile = jobs[job_type]["profile_wh"]
                finish = start + len(profile)

                required_setup = setup_slots if position else 0

                # Reject overlaps, missing setup time or late completion.
                if (
                    start < previous_finish + required_setup
                    or finish > len(prices)
                ):
                    break

                total_cost += sum(
                    energy * prices[start + offset]
                    for offset, energy in enumerate(profile)
                )

                previous_finish = finish

            else:
                valid_plans += 1
                best_cost = min(best_cost, total_cost)

    return best_cost, valid_plans

In [ ]:
test_cases = [
    ("Variable prices, no setup", test_prices, 0),
    ("Variable prices, 5-min setup", test_prices, 5),
    ("Variable prices, 10-min setup", test_prices, 10),
    ("Insufficient capacity", test_prices, 15),
    ("Constant prices", [0.00005] * 9, 5),
    (
        "Mixed positive and negative prices",
        [
            price if index % 2 else -price
            for index, price in enumerate(test_prices)
        ],
        5,
    ),
]

validation_rows = []

for name, prices, preparation_minutes in test_cases:
    exhaustive_cost, valid_count = exhaustive_minimum(
        test_jobs, prices, preparation_minutes
    )

    dp_result = optimize_day(
        job_types=test_jobs,
        prices_eur_per_wh=prices,
        setup_minutes=preparation_minutes,
    )

    dp_cost = dp_result["optimized_cost_eur"]

    if not isfinite(exhaustive_cost):
        assert dp_cost is None
    else:
        assert dp_cost is not None
        assert abs(dp_cost - exhaustive_cost) < 0.00000001

    validation_rows.append({
        "test": name,
        "valid_plans_enumerated": valid_count,
        "result": "PASS",
    })

validation_results_df = pd.DataFrame(validation_rows)
display(validation_results_df)

validation_results_df.to_csv(
    "algorithm_validation_results.csv",
    index=False,
)

print("All six independent validation tests passed.")